# Stage 4 — Per-frame feature extraction

Turns each video's ROI crops (rocket + exhaust plume) into a fixed-length
numeric vector **per frame**. These vectors are the training sequences for the
temporal anomaly model (stage 5) — training happens on *normal-only* footage,
so nothing here depends on failure examples.

**Inputs** (produced by stage 3, `scripts/roi_track.py`):
- `data/frames/{video}/roi/frame_*.jpg` — tracked rocket + plume crops
- `data/frames/{video}/roi_boxes.json` — tracking boxes + per-frame fallback flags
- `data/frames/{video}/manifest.json` — frame timestamps + lighting tag

**Outputs** (written by this stage, both cached to disk):
- `data/features/{video}.npy` — float32 array, shape `(n_frames, 33)`
- `data/features/{video}.frame_index.json` — row → frame file / timestamp /
  fallback flag, plus the video's metadata (source, mission, lighting, label)

**The 33 features** (defined in `scripts/features.py`):

| Group | Dims | Feature |
|---|---|---|
| Plume shape | 5 | area fraction, aspect ratio, mirror symmetry, centroid x / y |
| Plume contrast | 1 | core brightness vs surrounding annulus, in [-1, 1] |
| HSV histogram | 24 | coarse hue (12) + saturation (6) + value (6) bins, normalized |
| Edge / debris | 1 | Canny edge density **outside** the plume mask |
| Optical flow | 2 | mean and p95 Farneback flow magnitude / frame diagonal |

Values are made relative/normalized where possible so day and night footage
stay comparable (plume brightness is contrast against the local background,
not an absolute gray level).


In [ ]:
# --- Configuration (edit this cell for your environment) ---
import os, sys

IN_COLAB = False
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Drive folder that holds this repo's data/ directory.
    DRIVE_BASE = '/content/drive/MyDrive/rocket-launch-anomaly-detector'
    LOCAL = '/content/rocket-launch-anomaly-detector'   # Colab local disk
else:
    # Local run: both point at the repo root (where this notebook lives).
    DRIVE_BASE = LOCAL = os.path.abspath('.')

FRAMES = f'{LOCAL}/data/frames'          # ROI crops to process
FEATURES = f'{LOCAL}/data/features'      # where .npy outputs are written
FEATURES_DRIVE = f'{DRIVE_BASE}/data/features'  # push-back target on Drive

print('IN_COLAB =', IN_COLAB)
print('FRAMES   =', FRAMES)
print('FEATURES =', FEATURES)


### 1. Environment setup

On Colab this clones the repo (if it isn't there yet) and installs
dependencies. On a local checkout it just adds `scripts/` to the import path.
Run once per session.


In [ ]:
# --- Cell: install deps + make scripts/ importable ---
import shutil
import time

if not os.path.exists(f'{LOCAL}/scripts/features.py'):
    # Repo is missing or stale. Either set GIT_URL to your repo, or upload the
    # project (scripts/ + requirements.txt) into LOCAL manually and re-run.
    GIT_URL = 'https://github.com/YOUR_USERNAME/rocket-launch-anomaly-detector.git'
    if os.path.isdir(LOCAL) and os.listdir(LOCAL):
        stale = f'{LOCAL}_stale_{int(time.time())}'
        shutil.move(LOCAL, stale)
        print(f'moved existing {LOCAL} -> {stale}')
    elif os.path.isdir(LOCAL):
        os.rmdir(LOCAL)   # empty dir left by a previously failed clone
    print('cloning repo ...')
    !git clone --depth 1 {GIT_URL} "{LOCAL}"
    if not os.path.exists(f'{LOCAL}/scripts/features.py'):
        raise RuntimeError('clone produced no scripts/ — set GIT_URL to your repo '
                           'or upload the project into LOCAL')

!pip install -q -r "{LOCAL}/requirements.txt"

sys.path.insert(0, f'{LOCAL}/scripts')
import features as F

print('imported features; feature dims =', F.N_FEATURES)
print('feature names:', ', '.join(F.FEATURE_NAMES))


### 2. Mount Drive, copy frames locally, define push-back

Google Drive mounts are slow for many small reads, so `data/frames` is copied
once to the Colab local disk and all heavy I/O happens there. Completed
feature files are pushed back to Drive **per video**, so a mid-batch timeout
never loses work — just rerun the batch cell to resume.


In [ ]:
# --- Cell: mount Drive, copy frames locally, define per-video push-back ---
if IN_COLAB:
    drive.mount('/content/drive')

import shutil

os.makedirs(f'{LOCAL}/data', exist_ok=True)
if not os.path.isdir(FRAMES):
    print('copying data/frames to local disk ...')
    !cp -r "{DRIVE_BASE}/data/frames" "{LOCAL}/data/"
    print('copy done')
else:
    print('frames already on local disk:', FRAMES)


def push_features(video_name):
    """Copy one video's feature files back to Drive right after it finishes."""
    for f in (f'{video_name}.npy', f'{video_name}.frame_index.json'):
        src = os.path.join(FEATURES, f)
        if os.path.exists(src):
            os.makedirs(FEATURES_DRIVE, exist_ok=True)
            shutil.copy2(src, os.path.join(FEATURES_DRIVE, f))
            print('pushed', f)
    print('done:', video_name)


### 3. Run the feature batch (resumable)

Any video whose `.npy` already exists on Drive (or locally) is skipped, so
rerunning this one cell resumes an interrupted batch. Test on a single video
first via `ONLY`, then set it back to `None` for the full run.


In [ ]:
# --- Cell: run the feature batch (rerun this cell alone to resume) ---
# Videos whose .npy already exists on Drive are skipped, so an interrupted
# batch resumes cleanly. `after_video` pushes each result to Drive immediately.
done = set()
if os.path.isdir(FEATURES_DRIVE):
    done = {f[:-4] for f in os.listdir(FEATURES_DRIVE) if f.endswith('.npy')}
print('already on Drive:', sorted(done) or '(none)')

# Test on one video first, then set back to None for the full batch.
ONLY = None   # e.g. 'spacex_starlink-20230303'

F.batch(FRAMES, FEATURES, only=ONLY, skip=done, after_video=push_features)


### 4. Sanity-check the outputs

Every saved `.npy` must line up with its `.frame_index.json`: same number of
rows, the declared feature dimension, float32 dtype, and contiguous row
indices. This cell asserts all of that and prints a summary table.


In [ ]:
# --- Cell: verify every .npy / frame_index pair is consistent ---
import json
import numpy as np
from pathlib import Path

FEATURES_P = Path(FEATURES)
npys = sorted(FEATURES_P.glob('*.npy'))
print(f'found {len(npys)} feature files in {FEATURES}\n')
print(f'{"video":<32}{"frames":>8}{"feats":>6}{"fb":>5}  {"ok"}')
print('-' * 62)

all_ok = True
for npy in npys:
    video = npy.stem
    idx = json.loads(Path(str(npy).replace('.npy', '.frame_index.json')).read_text())
    X = np.load(npy)
    ok = (
        X.ndim == 2
        and X.dtype == np.float32
        and X.shape[1] == idx['n_features'] == F.N_FEATURES
        and len(idx['frames']) == X.shape[0]
        and [fr['row'] for fr in idx['frames']] == list(range(len(idx['frames'])))
    )
    all_ok &= bool(ok)
    fallback = sum(1 for fr in idx['frames'] if fr['fallback'])
    print(f'{video:<32}{X.shape[0]:>8}{X.shape[1]:>6}{fallback:>5}  {ok}')

print('\nALL CONSISTENT' if all_ok else '\nINCONSISTENCIES FOUND — investigate before training')


### 5. Inspect the features

Two quick visual checks before moving to stage 5: (1) the plume mask that the
shape/contrast features are computed from, and (2) a few feature traces over
the length of a launch video — they should be smooth and physically
meaningful, with no abrupt spikes unless something in the footage changed.


In [ ]:
# --- Cell: look at the plume mask a crop contributes ---
import cv2
import matplotlib.pyplot as plt

npys = sorted(Path(FEATURES).glob('*.npy'))
video = npys[0].stem
idx = json.loads(Path(str(npys[0]).replace('.npy', '.frame_index.json')).read_text())
mid = idx['frames'][len(idx['frames']) // 2]

crop = cv2.imread(f'{FRAMES}/{video}/roi/{mid["file"]}')
gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
mask = F.plume_mask(gray)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
ax[0].set_title(f'{video} :: {mid["file"]} (fallback={mid["fallback"]})')
ax[0].axis('off')
ax[1].imshow(gray, cmap='gray')
ax[1].imshow(mask, cmap='jet', alpha=0.35)
ax[1].set_title('plume mask (from plume_mask)')
ax[1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell: plot a few feature traces for one video ---
import numpy as np
import matplotlib.pyplot as plt

npys = sorted(Path(FEATURES).glob('*.npy'))
video = npys[0].stem
idx = json.loads(Path(str(npys[0]).replace('.npy', '.frame_index.json')).read_text())
X = np.load(npys[0])
names = idx['feature_names']

trace_cols = ['plume_area_frac', 'plume_symmetry', 'plume_bg_contrast',
              'edge_density_outside', 'flow_mean_mag', 'flow_p95_mag']
j = {n: names.index(n) for n in trace_cols}

fig, axes = plt.subplots(len(trace_cols), 1, figsize=(13, 2.3 * len(trace_cols)),
                         sharex=True)
for ax, n in zip(axes, trace_cols):
    ax.plot(np.arange(len(X)), X[:, j[n]], lw=0.8)
    ax.set_ylabel(n, fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel(f'frame ({video}, {len(X)} frames)')
fig.suptitle(f'Feature traces — {video}', fontsize=11)
plt.tight_layout()
plt.show()
